# RodPrep — exploration

Notebook d'exploration de l'étape 1 : extraction du récap Excel ROD et construction de la table hôtel.

Objectif : visualiser les entrées, les étapes intermédiaires et remplir `../Output/`.

In [51]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

ROOT = Path.cwd().resolve()
while ROOT.name != "RodPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Entrée — récap Excel et registre identité

In [52]:
from rod_ia.config.settings import get_settings
from rod_ia.domain.repositories.identity_registry import HotelIdentityRegistry
from rod_prep.prep import RodPrep

settings = get_settings(PROJECT)
prep = RodPrep(INPUT_DIR, OUTPUT_DIR, settings.identity_registry_path)

recap_path = prep.seed_input_from_sources()
print("Fichier récap :", recap_path)

registry = HotelIdentityRegistry(settings.identity_registry_path)
registry_df = pd.DataFrame([r.to_dict() for r in registry.all_records()])
print(f"Registre identité : {len(registry_df)} hôtels")
display(registry_df.head(3))
registry_df[["hotel_id", "name_ventes", "brand", "city", "nb_chambres"]].head(10)

Fichier récap : /media/laghmari/ssd-data/dev/hotels/prepare/RodPrep/Input/recapitulatif_rod.xlsx
Registre identité : 8 hôtels


,hotel_id,brand,city,name_display,name_ventes,name_rod,aliases,lat_canonical,lon_canonical,geo_source,lat_rod,lon_rod,lat_nominatim,lon_nominatim,has_sales,has_rod,nb_chambres
0,ibis-budget-nice,IBIS BUDGET,Nice,Ibis budget Nice Californie,Ibis budget Nice,Nice Californie,"[Ibis Budget Nice, IBIS BUDGET Nice]",43.710000,7.260000,nominatim,None,None,43.689258,7.240379,True,True,129.0
1,ibis-budget-strasbourg,IBIS BUDGET,Strasbourg,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,Strasbourg République,"[Ibis Budget Strasbourg, Ibis budget Strasbourg]",NaN,NaN,None,None,None,NaN,NaN,True,True,97.0
2,ibis-styles-roissy-cdg,IBIS STYLES,Roissy,Ibis Styles Roissy CDG,None,Roissy CDG,[],49.007078,2.520403,nominatim,None,None,49.007078,2.520403,False,True,309.0


,hotel_id,name_ventes,brand,city,nb_chambres
0,ibis-budget-nice,Ibis budget Nice,IBIS BUDGET,Nice,129.0
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0
2,ibis-styles-roissy-cdg,None,IBIS STYLES,Roissy,309.0
3,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,Megève,572.0
4,novotel-paris-tour-eiffel,Novotel Paris Tour Eiffel,NOVOTEL,Paris,764.0
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,305.0
6,mercure-boulogne,None,MERCURE,Boulogne-Billancourt,191.0
7,novotel-porte-italie,Novotel Porte d'Italie,NOVOTEL,Paris,NaN


## 2. Extraction longue — une ligne par variable × hôtel

In [53]:
from rod_ia.domain.services.rod_recap_extractor import RodRecapExtractor

extractor = RodRecapExtractor(
    recap_path=recap_path,
    identity_registry=registry,
    output_path=OUTPUT_DIR / "rod_recap",
)

long_df = extractor.extract_long()
print(f"Format long : {long_df.shape[0]} lignes × {long_df.shape[1]} colonnes")
long_df.head(12)

Format long : 938 lignes × 9 colonnes


,hotel_id,recap_column,row,etape,sous_etape,data_label,field_key,field_type_hint,raw_value
0,ibis-budget-nice,NICE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H2075
1,ibis-budget-strasbourg,STRASBOURG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB6A3
2,ibis-styles-roissy-cdg,PARIS CDG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0815
3,novotel-megeve,MEGEVE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB5I0
4,novotel-paris-tour-eiffel,TOUR EIFFEL,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H3546
5,mercure-montmartre,MONTMARTRE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0373
6,mercure-boulogne,BOULOGNE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H6188
7,ibis-budget-nice,NICE,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET NICE CALIFORNIE
8,ibis-budget-strasbourg,STRASBOURG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET STRASBOURG REPUBLIQUE
9,ibis-styles-roissy-cdg,PARIS CDG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS STYLES ROISSY CDG


## 3. Format wide — features `d_recap_*` par hôtel

In [54]:
wide_df = extractor.extract_wide()
print(f"Format wide : {wide_df.shape[0]} hôtels × {wide_df.shape[1]} colonnes")
wide_df.head()

Format wide : 7 hôtels × 91 colonnes


,hotel_id,d_recap_0_page_de_connexion_id_code_h,d_recap_0_page_de_connexion_id_nom_de_l_hotel,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_1_informations_generales_donnees_admin_contrat_signe_annee,d_recap_1_informations_generales_donnees_admin_contrat_type,d_recap_1_informations_generales_donnees_admin_derniere_reno_hotel,d_recap_1_informations_generales_donnees_admin_derniere_reno_lobby,d_recap_1_informations_generales_donnees_admin_dom_dof,d_recap_1_informations_generales_donnees_admin_marque,d_recap_1_informations_generales_donnees_admin_nb_de_chambres,d_recap_1_informations_generales_donnees_admin_pms,d_recap_1_informations_generales_donnees_admin_proprietaire,d_recap_1_informations_generales_donnees_chiffrees_to_annuel,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_taux,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_taux,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_caf,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_r_frig_r_e,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygi_ne,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pr_t_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosm_tiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_4_informations_corner_autre_emplacement_dispo_m_tres_carr_s_estimation,d_recap_4

## 4. Table de liaison `hotel_lookup`

In [55]:
hotel_lookup = prep.run()  # persiste aussi rod_features + hotel_lookup
print(f"hotel_lookup : {hotel_lookup.shape}")
hotel_lookup.head()

hotel_lookup : (8, 98)


,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,hotel_lat,hotel_lon,nb_chambres,d_recap_0_page_de_connexion_id_code_h,d_recap_0_page_de_connexion_id_nom_de_l_hotel,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_1_informations_generales_donnees_admin_contrat_signe_annee,d_recap_1_informations_generales_donnees_admin_contrat_type,d_recap_1_informations_generales_donnees_admin_derniere_reno_hotel,d_recap_1_informations_generales_donnees_admin_derniere_reno_lobby,d_recap_1_informations_generales_donnees_admin_dom_dof,d_recap_1_informations_generales_donnees_admin_marque,d_recap_1_informations_generales_donnees_admin_nb_de_chambres,d_recap_1_informations_generales_donnees_admin_pms,d_recap_1_informations_generales_donnees_admin_proprietaire,d_recap_1_informations_generales_donnees_chiffrees_to_annuel,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_taux,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_taux,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_caf,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_r_frig_r_e,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygi_ne,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pr_t_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosm_tiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_4_

## 5. Aperçu colonnes récap retenues

In [56]:
recap_cols = [c for c in hotel_lookup.columns if str(c).startswith("d_recap_")]
print(f"{len(recap_cols)} colonnes d_recap_")
if recap_cols:
    hotel_lookup[["hotel_code", "nom_hotel"] + recap_cols[:8]].head()

90 colonnes d_recap_


## 6. Entrée MeteoPrep / ProximityPrep

In [57]:
meteo_input = prep.to_meteo_input()
meteo_input

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,ibis-budget-nice,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689258,7.240379
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,NaN,NaN
2,ibis-styles-roissy-cdg,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.007078,2.520403
3,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859850,6.619478
4,novotel-paris-tour-eiffel,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.850000,2.350000
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,NaN,NaN
6,mercure-boulogne,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.843936,2.230002
7,novotel-porte-italie,Novotel Porte d'Italie,NOVOTEL,Paris,NaN,NaN


## 7. Fichiers produits dans Output/

In [58]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

hotel_lookup.csv
hotel_lookup.parquet
rod_features.csv
rod_features.parquet
rod_recap.long.csv
rod_recap.schema.json
rod_recap.wide.csv


In [59]:
hotel_lookup

,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,hotel_lat,hotel_lon,nb_chambres,d_recap_0_page_de_connexion_id_code_h,d_recap_0_page_de_connexion_id_nom_de_l_hotel,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_1_informations_generales_donnees_admin_contrat_signe_annee,d_recap_1_informations_generales_donnees_admin_contrat_type,d_recap_1_informations_generales_donnees_admin_derniere_reno_hotel,d_recap_1_informations_generales_donnees_admin_derniere_reno_lobby,d_recap_1_informations_generales_donnees_admin_dom_dof,d_recap_1_informations_generales_donnees_admin_marque,d_recap_1_informations_generales_donnees_admin_nb_de_chambres,d_recap_1_informations_generales_donnees_admin_pms,d_recap_1_informations_generales_donnees_admin_proprietaire,d_recap_1_informations_generales_donnees_chiffrees_to_annuel,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_bas_taux,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_mois,d_recap_1_informations_generales_donnees_chiffrees_to_le_plus_haut_taux,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_caf,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_r_frig_r_e,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolis_es,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sal_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucr_s_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygi_ne,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pr_t_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosm_tiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_4_